# 01 Data Processing

This notebook rebuilds the panel from the current raw inputs in `data/raw`:

- `2000-2025-bm-ir-infl.xlsx` for the base macro country-year panel and exchange rate
- `2000-2023-hc-pop.csv` for human capital and population
- `2000-2025-lending-deposit-tourism-arrivals.csv` for deposit rate, lending rate, and tourism arrivals

The active processing window is `2000-2023`, and the notebook now delegates the ingestion logic to `src.reprocess_current_raw` so the notebook and exported CSV stay on the same contract.


## Setup

Load the shared preprocessing functions and inspect the source files before rebuilding the panel.


In [51]:
from pathlib import Path
import sys
import importlib

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import (
    CURRENT_ADDITIONAL_SERIES_FILE,
    CURRENT_HC_POP_FILE,
    CURRENT_MACRO_PANEL_WORKBOOK,
    OUTPUTS_DIR,
    PROCESSED_PANEL_FILE,
    TIME_WINDOW,
    ensure_output_dirs,
)
import src.reprocess_current_raw as reprocess_current_raw
importlib.reload(reprocess_current_raw)

build_clean_panel = reprocess_current_raw.build_clean_panel
load_additional_series_panel = reprocess_current_raw.load_additional_series_panel
load_hc_pop_panel = reprocess_current_raw.load_hc_pop_panel
load_macro_panel = reprocess_current_raw.load_macro_panel
save_outputs = reprocess_current_raw.save_outputs

ensure_output_dirs()
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## Source Audit

Confirm the three active sources and their effective coverage in the processing window.


In [52]:
macro_df = load_macro_panel()
hc_df = load_hc_pop_panel()
additional_df = load_additional_series_panel()

source_audit = pd.DataFrame(
    [
        {
            "file": CURRENT_MACRO_PANEL_WORKBOOK.name,
            "rows": len(macro_df),
            "countries": macro_df["country_code"].nunique(),
            "first_year": int(macro_df["year"].min()),
            "last_year": int(macro_df["year"].max()),
        },
        {
            "file": CURRENT_HC_POP_FILE.name,
            "rows": len(hc_df),
            "countries": hc_df["country_code"].nunique(),
            "first_year": int(hc_df["year"].min()),
            "last_year": int(hc_df["year"].max()),
        },
        {
            "file": CURRENT_ADDITIONAL_SERIES_FILE.name,
            "rows": len(additional_df),
            "countries": additional_df["country_code"].nunique(),
            "first_year": int(additional_df["year"].min()),
            "last_year": int(additional_df["year"].max()),
        },
    ]
)

print(f"Configured TIME_WINDOW: {TIME_WINDOW}")
source_audit


Configured TIME_WINDOW: (2000, 2023)


,file,rows,countries,first_year,last_year
0,2000-2025-bm-ir-infl.xlsx,264,11,2000,2023
1,2000-2023-hc-pop.csv,240,10,2000,2023
2,2000-2025-lending-deposit-tourism-arrivals.csv,229,10,2000,2023


In [53]:
display_columns = ["country_code", "country", "year"]

print("Macro panel sample")
display(
    macro_df[
        display_columns + ["fdi_pct_gdp", "broad_money_growth_pct", "real_interest_rate_pct"]
    ].head()
)

print("HC / population sample")
display(hc_df[display_columns + ["hc_human_capital_index", "population_total"]].head())

print("Supplementary series sample")
display(
    additional_df[
        display_columns + ["deposit_interest_rate_pct", "lending_interest_rate_pct", "tourism_arrivals"]
    ].head()
)


Macro panel sample


,country_code,country,year,fdi_pct_gdp,broad_money_growth_pct,real_interest_rate_pct
0,BRN,Brunei Darussalam,2000,8.3641,47.6608,3.0508
1,BRN,Brunei Darussalam,2001,0.9956,-11.9304,11.0211
2,BRN,Brunei Darussalam,2002,3.6265,1.8854,5.6430
3,BRN,Brunei Darussalam,2003,1.7275,4.0711,-0.7619
4,BRN,Brunei Darussalam,2004,1.3134,15.8347,-9.4747


HC / population sample


,country_code,country,year,hc_human_capital_index,population_total
0,BRN,Brunei Darussalam,2000,2.6113,"326,424.0000"
1,BRN,Brunei Darussalam,2001,2.6180,"333,345.0000"
2,BRN,Brunei Darussalam,2002,2.6247,"340,099.0000"
3,BRN,Brunei Darussalam,2003,2.6314,"346,637.0000"
4,BRN,Brunei Darussalam,2004,2.6381,"352,911.0000"


Supplementary series sample


,country_code,country,year,deposit_interest_rate_pct,lending_interest_rate_pct,tourism_arrivals
0,BRN,Brunei Darussalam,2000,NaN,5.5000,"984,000.0000"
1,BRN,Brunei Darussalam,2001,NaN,5.5000,"840,000.0000"
2,BRN,Brunei Darussalam,2002,NaN,5.5000,"891,000.0000"
3,BRN,Brunei Darussalam,2003,1.0648,5.5000,"944,000.0000"
4,BRN,Brunei Darussalam,2004,1.0439,5.5000,NaN


## Build Clean Panel

Run the shared reprocessor, inspect the resulting panel, and then write the refreshed outputs.


In [54]:
clean_panel, outputs = build_clean_panel()

panel_overview = pd.DataFrame(
    {
        "metric": ["rows", "countries", "first_year", "last_year"],
        "value": [
            len(clean_panel),
            clean_panel["country"].nunique(),
            int(clean_panel["year"].min()),
            int(clean_panel["year"].max()),
        ],
    }
)

panel_overview


,metric,value
0,rows,264
1,countries,11
2,first_year,2000
3,last_year,2023


In [55]:
clean_panel.head()


,country_id,country_code,country,year,fdi_pct_gdp,fdi_pct_gdp_winsorized,broad_money_growth_pct,trade_pct_gdp,inflation_gdp_deflator_pct,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct,hc_human_capital_index,ln_gdppc,xr_dep_pct,xr_dep_pct_winsorized,ln_population_total,ln_tourism_arrivals
0,1,BRN,Brunei Darussalam,2000,8.3641,8.3641,47.6608,87.0553,2.3767,NaN,3.0508,5.5000,2.6113,22.6059,NaN,NaN,12.6960,13.7994
1,1,BRN,Brunei Darussalam,2001,0.9956,0.9956,-11.9304,91.0998,-4.9731,NaN,11.0211,5.5000,2.6180,22.5309,3.8552,3.8552,12.7169,13.6412
2,1,BRN,Brunei Darussalam,2002,3.6265,3.6265,1.8854,93.2162,-0.1354,NaN,5.6430,5.5000,2.6247,22.5691,-0.0633,-0.0633,12.7370,13.7001
3,1,BRN,Brunei Darussalam,2003,1.7275,1.7275,4.0711,89.0856,6.3100,1.0648,-0.7619,5.5000,2.6314,22.6929,-2.7405,-2.7405,12.7560,13.7579
4,1,BRN,Brunei Darussalam,2004,1.3134,1.3134,15.8347,84.8822,16.5420,1.0439,-9.4747,5.5000,2.6381,22.8773,-3.0275,-3.0275,12.7740,13.6844


In [56]:
clean_panel.notna().sum().rename("non_missing").to_frame()


,non_missing
country_id,264
country_code,264
country,264
year,264
fdi_pct_gdp,260
fdi_pct_gdp_winsorized,260
broad_money_growth_pct,240
trade_pct_gdp,240
inflation_gdp_deflator_pct,264
deposit_interest_rate_pct,213


In [57]:
save_outputs(
    clean_panel=clean_panel,
    winsorization_thresholds=outputs["winsorization_thresholds"],
    control_imputation_log=outputs["control_imputation_log"],
    transformation_audit=outputs["transformation_audit"],
    coverage_by_variable=outputs["coverage_by_variable"],
    coverage_by_country=outputs["coverage_by_country"],
    review_flags=outputs["review_flags"],
    variable_audit=outputs["variable_audit"],
    raw_input_audit=outputs["raw_input_audit"],
)

print(f"Saved processed panel to {PROCESSED_PANEL_FILE}")
print(f"Saved preprocessing workbook to {OUTPUTS_DIR / 'preprocessing_outputs.xlsx'}")


Saved processed panel to /Users/bunnypro/Projects/monetary_policy_fdi_analysis/data/processed/clean_panel.csv
Saved preprocessing workbook to /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/preprocessing_outputs.xlsx


## Processing Diagnostics

Review the export-side audit tables that downstream notebooks rely on.


In [58]:
outputs["raw_input_audit"]


,file_name,role,years,countries,rows,notes
0,2000-2025-bm-ir-infl.xlsx,base_macro_panel,2000-2023,11,264,Uses WDI-style country-year rows directly from...
1,2000-2023-hc-pop.csv,human_capital_population_panel,2000-2023,10,240,"Melts row-by-variable yearly columns, uses onl..."
2,2000-2025-lending-deposit-tourism-arrivals.csv,supplementary_interest_tourism_panel,2000-2023,10,229,"Melts year columns and merges deposit rate, le..."


In [59]:
outputs["variable_audit"].sort_values(["missing_rate", "variable"], ascending=[False, True]).reset_index(drop=True)


,variable,non_missing,total_rows,missing,missing_rate,recommended_handling
0,lending_interest_rate_pct,192,264,72,0.2727,processed_from_current_raw_inputs
1,real_interest_rate_pct,205,264,59,0.2235,processed_from_current_raw_inputs
2,deposit_interest_rate_pct,213,264,51,0.1932,processed_from_current_raw_inputs
3,broad_money_growth_pct,240,264,24,0.0909,processed_from_current_raw_inputs
4,hc_human_capital_index,240,264,24,0.0909,leave_missing_and_report_limitation
5,ln_population_total,240,264,24,0.0909,processed_from_current_raw_inputs
6,ln_tourism_arrivals,240,264,24,0.0909,processed_from_current_raw_inputs
7,trade_pct_gdp,240,264,24,0.0909,processed_from_current_raw_inputs
8,xr_dep_pct,253,264,11,0.0417,processed_from_current_raw_inputs
9,xr_dep_pct_winsorized,253,264,11,0.0417,processed_from_current_raw_inputs


In [60]:
outputs["control_imputation_log"]


,variable,handling_applied,missing_before,missing_after,filled_values
0,trade_pct_gdp,within_country_interpolate_then_edge_fill,31,24,7
1,inflation_gdp_deflator_pct,within_country_interpolate_then_edge_fill,0,0,0
2,ln_gdppc,within_country_interpolate_then_edge_fill,0,0,0
3,xr_dep_pct,within_country_interpolate_and_edge_fill_excep...,14,11,3
4,ln_population_total,within_country_interpolate_then_edge_fill,24,24,0
5,ln_tourism_arrivals,within_country_interpolate_then_edge_fill_keep...,64,24,40
6,hc_human_capital_index,leave_missing_and_report_limitation,24,24,0


In [61]:
outputs["review_flags"]


,country,year,flag,value,note
1,Myanmar,2002,inflation_extreme_keep_and_review,41.5089,Observed value retained for main analysis.
2,Timor-Leste,2021,inflation_extreme_keep_and_review,59.0797,Observed value retained for main analysis.
3,Viet Nam,2010,inflation_extreme_keep_and_review,42.3033,Observed value retained for main analysis.
0,Myanmar,2012,xr_dep_pct_extreme_keep_and_review,476.7955,Observed value retained; winsorized version is...
